# 재순위화 측정 (Colab GPU)

로컬 CPU 기준 질의당 25.16초, 220건에 약 92분. GPU로 옮겨 돌린다.

**실행 순서**

1. 우측 상단 `Select Kernel` → `Colab` → GPU(T4)
2. 탐색기에서 `data/pairs-intent.json.gz` 우클릭 → **Upload to Colab**
3. Run All

`google.colab.files.upload()` 위젯은 VS Code 확장에서 동작하지 않으므로 2번으로 대신한다.
브라우저 Colab에서 돌릴 때는 좌측 파일 탭에 끌어다 놓으면 된다.

업로드한 파일에 `intent_filter`가 있으면 결과 파일명이 `rerank-intent-*`, 없으면 `rerank-dense-*`가 된다.

| 셀 | 내용 |
|---|---|
| 1 | 환경 확인 + 설치 |
| 2 | `pairs*.json(.gz)` 탐색 및 적재 |
| 3 | 지표 계산 함수 |
| 4 | `bge-reranker-base` (논문) |
| 5 | `bge-reranker-v2-m3-ko` — ADR-003에서 기각, 기본 건너뜀 |
| 6 | 결과 저장 및 요약 |

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

!pip -q install sentence-transformers

In [ ]:
import glob, gzip, json, os

# VS Code Colab 확장은 files.upload() 위젯 미지원
# 탐색기에서 data/pairs*.json.gz 우클릭 -> Upload to Colab
def find_pairs():
    hits = []
    for root in ('.', '/content', '/content/data'):
        hits += glob.glob(os.path.join(root, 'pairs*.json*'))
    hits += glob.glob('/content/**/pairs*.json*', recursive=True)
    return sorted({os.path.realpath(h) for h in hits})

found = find_pairs()
if not found:
    raise SystemExit('pairs 파일 없음. 탐색기에서 우클릭 -> Upload to Colab')
if len(found) > 1:
    print('여러 개 발견, 첫 번째 사용:', found)
path = found[0]

with (gzip.open if path.endswith('.gz') else open)(path, 'rt', encoding='utf-8') as f:
    data = json.load(f)

queries = data['queries']
TAG = 'intent' if data.get('intent_filter') else 'dense'
print(f"{path}")
print(f"질의 {len(queries)} / 쌍 {sum(len(q['docs']) for q in queries)} / 인덱스 {data['index']} / 후보 {TAG}")

In [ ]:
import statistics, time

CUTS = (5, 10, 20)

# order_fn 반환값: (재정렬된 문서, 점수). 점수는 폴백 임계값 산출용
def evaluate(order_fn, label):
    prec = {c: [] for c in CUTS}
    intent = {c: [] for c in CUTS}
    disease = {c: [] for c in CUTS}
    rr, per_query = [], []
    t0 = time.time()

    for i, q in enumerate(queries, 1):
        docs, scores = order_fn(q)
        hits = [d['disease_name'] == q['gold_disease'] and d['intention'] == q['gold_intention'] for d in docs]
        rr.append(1 / (hits.index(True) + 1) if True in hits else 0.0)

        row = {'source': q['source'], 'intention': q['gold_intention'], 'disease_name': q['gold_disease'],
               'precision': {}, 'intention_rate': {}, 'disease_rate': {}, 'rr': rr[-1],
               'scores': [round(float(s), 5) for s in scores], 'hits': hits}
        for c in CUTS:
            top = docs[:c]
            p = sum(hits[:c]) / len(top)
            ir = sum(d['intention'] == q['gold_intention'] for d in top) / len(top)
            dr = sum(d['disease_name'] == q['gold_disease'] for d in top) / len(top)
            prec[c].append(p); intent[c].append(ir); disease[c].append(dr)
            row['precision'][f'@{c}'] = p
            row['intention_rate'][f'@{c}'] = ir
            row['disease_rate'][f'@{c}'] = dr
        per_query.append(row)

        if i % 50 == 0:
            print(f'  {i}/{len(queries)} ({time.time()-t0:.0f}s)')

    return {
        'label': label, 'n': len(per_query),
        'precision': {f'@{c}': statistics.mean(prec[c]) for c in CUTS},
        'intention_rate': {f'@{c}': statistics.mean(intent[c]) for c in CUTS},
        'disease_rate': {f'@{c}': statistics.mean(disease[c]) for c in CUTS},
        'mrr': statistics.mean(rr),
        'elapsed_sec': round(time.time() - t0, 1),
        'per_query': per_query,
    }

def show(r):
    print(f"\n{r['label']}  (질의 {r['n']}, {r['elapsed_sec']}s)")
    print(f"  {'':<14}{'@5':>9}{'@10':>9}{'@20':>9}")
    for k, name in (('precision','Precision'), ('intention_rate','의도'), ('disease_rate','질환')):
        print(f"  {name:<14}" + ''.join(f"{r[k][f'@{c}']:>9.4f}" for c in CUTS))
    print(f"  MRR {r['mrr']:.4f}")

    # 정답/오답 질의의 top1 점수 분리 여부 확인
    hit = [q['scores'][0] for q in r['per_query'] if q['hits'][0]]
    miss = [q['scores'][0] for q in r['per_query'] if not q['hits'][0]]
    for name, xs in (('top1 정답', hit), ('top1 오답', miss)):
        if xs:
            print(f"  {name:<10} n={len(xs):<4} 중앙값 {statistics.median(xs):>8.3f}  "
                  f"범위 [{min(xs):.3f}, {max(xs):.3f}]")

In [ ]:
from sentence_transformers import CrossEncoder

def make_reranker(model_name, batch_size=128):
    ce = CrossEncoder(model_name, max_length=512, device='cuda')
    def order(q):
        pairs = [[q['question'], d['text']] for d in q['docs']]
        scores = ce.predict(pairs, batch_size=batch_size, show_progress_bar=False)
        ranked = sorted(zip(q['docs'], scores), key=lambda x: x[1], reverse=True)
        return [d for d, _ in ranked], [s for _, s in ranked]
    return order

results = {}
results['base'] = evaluate(make_reranker('BAAI/bge-reranker-base'), 'rerank(bge-reranker-base)')
show(results['base'])

In [ ]:
# ADR-003 에서 기각된 모델. 재확인이 필요할 때만 True
RUN_KO = False

if RUN_KO:
    results['ko'] = evaluate(make_reranker('dragonkue/bge-reranker-v2-m3-ko'), 'rerank(bge-reranker-v2-m3-ko)')
    show(results['ko'])
else:
    print('건너뜀 (RUN_KO=False)')

In [ ]:
import os

for key, r in results.items():
    fn = f'/content/rerank-{TAG}-{key}.json'
    json.dump(r, open(fn, 'w', encoding='utf-8'), ensure_ascii=False)
    print(f'{fn}  {os.path.getsize(fn)/1024:.0f}KB')

print()
print(f"{'':<28}{'P@5':>9}{'P@10':>9}{'P@20':>9}{'의도@5':>9}{'질환@5':>9}{'MRR':>9}")
for key, r in results.items():
    print(f"{key:<28}"
          + ''.join(f"{r['precision'][f'@{c}']:>9.4f}" for c in CUTS)
          + f"{r['intention_rate']['@5']:>9.4f}{r['disease_rate']['@5']:>9.4f}{r['mrr']:>9.4f}")

print()
print('Colab 사이드바 Contents 에서 우클릭 -> Download')